<a href="https://colab.research.google.com/github/catseyehuang/Taipei-YouBike-Dashboard/blob/main/YouBike_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#YouBike


## JSON 批次讀取與建立 DataFrame

In [1]:
# ==========================================
# 🚀 Data Lake JSON 批次讀取
# ==========================================

import os
import json
import pandas as pd
from google.colab import drive

# 1. 掛載您的 Google Drive (執行時會跳出視窗要求授權)
print("🔗 正在要求授權掛載 Google Drive...")
drive.mount('/content/drive')

# 2. 指定 Data Lake 資料夾路徑
folder_path = '/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON'

all_records = []
print(f"\n📂 正在掃描資料夾：{folder_path}")

# 3. 掃描並解析資料夾內的所有 JSON 檔案
if not os.path.exists(folder_path):
    print(f"❌ 找不到路徑 {folder_path}，請檢查資料夾名稱是否正確！")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".json")]
    print(f"🔍 共找到 {len(file_list)} 個 JSON 快照檔案，開始進行清洗與對齊...")

    for filename in file_list:
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)

                # 🛡️ 處理新北市格式 (NTPC)
                if filename.startswith("NTPC_"):
                    for item in data:
                        # 動態防禦欄位名稱
                        tot = int(item.get('tot_quantity', item.get('tot', item.get('Quantity', 0))))
                        if tot > 0:
                            all_records.append({
                                'city': 'New Taipei',
                                'sno': item.get('sno', ''),
                                'sna': item.get('sna', ''),
                                'sarea': item.get('sarea', ''),
                                'tot_quantity': tot,
                                'sbi_quantity': int(item.get('sbi_quantity', item.get('sbi', item.get('available_rent_bikes', 0)))),
                                'bemp': int(item.get('bemp', item.get('available_return_bikes', 0))),
                                'lat': float(item.get('lat', item.get('latitude', 0))),
                                'lng': float(item.get('lng', item.get('longitude', 0))),
                                'update_time': item.get('mday', item.get('updateTime', '')),
                                'source_file': filename # 追蹤來源檔案
                            })

                # 🛡️ 處理台北市格式 (TPE)
                elif filename.startswith("TPE_"):
                    for item in data:
                        tot = int(item.get('quantity', item.get('tot', item.get('Quantity', 0))))
                        if tot > 0:
                            all_records.append({
                                'city': 'Taipei',
                                'sno': item.get('sno', ''),
                                'sna': item.get('sna', ''),
                                'sarea': item.get('sarea', ''),
                                'tot_quantity': tot,
                                'sbi_quantity': int(item.get('available_rent_bikes', item.get('sbi', 0))),
                                'bemp': int(item.get('available_return_bikes', item.get('bemp', 0))),
                                'lat': float(item.get('latitude', item.get('lat', 0))),
                                'lng': float(item.get('longitude', item.get('lng', 0))),
                                'update_time': item.get('mday', item.get('updateTime', '')),
                                'source_file': filename
                            })
            except Exception as e:
                print(f"⚠️ 解析檔案 {filename} 發生錯誤: {e}")

    # 4. 轉換為 Pandas DataFrame
    if len(all_records) > 0:
        df_history = pd.DataFrame(all_records)

        # 時間格式標準化 (將字串轉為 datetime 物件，方便後續畫圖)
        df_history['update_time'] = pd.to_datetime(df_history['update_time'], format='mixed', errors='coerce')

        # 依照時間與站點排序，確保時序正確
        df_history = df_history.sort_values(by=['update_time', 'sno']).reset_index(drop=True)

        print(f"\n✅ 成功將 JSON 轉換為 DataFrame！共載入 {len(df_history)} 筆資料。")

        # 顯示資料全貌與前五筆
        print("="*45)
        display(df_history.info())
        print("="*45)
        #display(df_history.head())
    else:
        print("\n⚠️ 目錄中沒有符合格式的資料，請確認 GAS 排程是否有成功下載檔案。")

🔗 正在要求授權掛載 Google Drive...
Mounted at /content/drive

📂 正在掃描資料夾：/content/drive/MyDrive/C02_AI/Vibe Coding/YouBike/YouBike_Raw_JSON
🔍 共找到 608 個 JSON 快照檔案，開始進行清洗與對齊...

✅ 成功將 JSON 轉換為 DataFrame！共載入 1008976 筆資料。
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008976 entries, 0 to 1008975
Data columns (total 11 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   city          1008976 non-null  object        
 1   sno           1008976 non-null  object        
 2   sna           1008976 non-null  object        
 3   sarea         1008976 non-null  object        
 4   tot_quantity  1008976 non-null  int64         
 5   sbi_quantity  1008976 non-null  int64         
 6   bemp          1008976 non-null  int64         
 7   lat           1008976 non-null  float64       
 8   lng           1008976 non-null  float64       
 9   update_time   1008976 non-null  datetime64[ns]
 10  source_file   1008976 non-null  object        
dt

None

In [ ]:
# ==========================================
# 📊 原始 JSON 檔案統計透視表
# ==========================================

# 1. 從 source_file 欄位中提取日期
# 檔名格式為 例如: TPE_20260617_172106.json, 我們需要 '20260617'
df_history['file_date'] = df_history['source_file'].str.extract(r'[A-Z]{3}_(\d{8})_\d{6}\.json', expand=False)

# 新增 scrape_timestamp 和 scrape_time 欄位，確保其在後續計算中可用
df_history['scrape_timestamp'] = df_history['source_file'].str.extract(r'_(\d{8}_\d{6})\.json')
df_history['scrape_time'] = pd.to_datetime(df_history['scrape_timestamp'], format='%Y%m%d_%H%M%S', errors='coerce')

# 2. 創建一個簡化的 'city_type' 欄位，用於作為 pivot table 的欄位
df_history['city_type'] = df_history['city'].map({'Taipei': 'TPE', 'New Taipei': 'NTPC'})

# 3. 計算每個日期和城市類型的檔案筆數 (其實是記錄筆數)
# 使用 source_file 和 file_date 確保每個原始檔案只被計算一次
# 這裡我們需要統計的是 '有幾筆檔案' 而不是 df_history 中的行數，
# 所以先對 source_file, city_type, file_date 進行去重複，然後再計數
file_counts = df_history[['source_file', 'city_type', 'file_date']].drop_duplicates()

pivot_table_df = file_counts.pivot_table(
    index='file_date',       # 左側垂直欄位：根據檔名上的時間戳日期
    columns='city_type',     # 上方水平欄位：TPE, NTPC
    aggfunc='size',          # 數值：統計有幾筆檔案 (這裡的'檔案'是指'source_file'的唯一值)
    fill_value=0             # 缺失值填充為0
)

# 確保日期是排序的
pivot_table_df = pivot_table_df.sort_index()

print("📈 原始 JSON 檔案統計透視表 (根據檔名日期和城市):")
display(pivot_table_df)

In [ ]:
df_history['scrape_hour'] = df_history['scrape_time'].dt.hour

# Filter for 2026-06-17
df_20260617 = df_history[df_history['file_date'] == '20260617'].copy()

# Group by scrape_hour and city_type to count unique source_files
hourly_file_counts = df_20260617.groupby(['scrape_hour', 'city_type'])['source_file'].nunique().unstack(fill_value=0)

# Ensure all 24 hours are present, filling missing hours with 0
hourly_file_counts = hourly_file_counts.reindex(range(24), fill_value=0)

print("📈 2026年6月17日 每小時 JSON 檔案數量 (根據檔名日期和城市):")
display(hourly_file_counts)

In [ ]:
# Ensure 'scrape_hour' is available (already created in previous cell)
df_history['scrape_hour'] = df_history['scrape_time'].dt.hour

# Group by file_date, scrape_hour, and city_type to count unique source_files
daily_hourly_file_counts = df_history.groupby(['file_date', 'scrape_hour', 'city_type'])['source_file'].nunique().unstack(fill_value=0)

# Reindex to ensure all 24 hours are present for each date
# Create a MultiIndex for all possible date-hour combinations
all_dates = df_history['file_date'].unique()
all_hours = range(24)

# Create a complete MultiIndex with all date-hour combinations
full_idx = pd.MultiIndex.from_product([all_dates, all_hours], names=['file_date', 'scrape_hour'])

# Reindex the grouped data to fill in missing hours with 0
daily_hourly_file_counts = daily_hourly_file_counts.reindex(full_idx, fill_value=0)

# Sort by date and hour for better readability
daily_hourly_file_counts = daily_hourly_file_counts.sort_index()

print("📈 每日每小時 JSON 檔案數量 (根據檔名日期和城市):")
display(daily_hourly_file_counts)

In [ ]:
# 由於 NTPC 和 TPE 的檔案數量相同，我們取其中一個城市 (例如 NTPC) 的資料
# 並將 file_date 從索引層級 unstack 到欄位層級
upgraded_pivot_table = daily_hourly_file_counts['NTPC'].unstack(level='file_date')

print("📈 每日每小時 JSON 檔案數量 (優化版，每天為欄位，每小時為列，單城市檔案筆數):")
display(upgraded_pivot_table)

In [ ]:
import plotly.express as px

# Create the heatmap using the upgraded_pivot_table
fig_heatmap = px.imshow(
    upgraded_pivot_table.values,
    x=upgraded_pivot_table.columns.astype(str),
    y=upgraded_pivot_table.index,
    labels=dict(x="檔案日期 (file_date)", y="撈取小時 (scrape_hour)", color="檔案數量"),
    title="每日每小時 JSON 檔案數量熱圖 (單城市)",
    color_continuous_scale='pubu' # Changed color scale for better zero visibility
)

fig_heatmap.update_xaxes(side="top")

fig_heatmap.show()

NameError: name 'upgraded_pivot_table' is not defined

##🧹 資料淨化：建立黑名單與幽靈站點剔除

In [2]:
# ==========================================
# 🧹 資料淨化：列級過濾 (剔除特定異常時間點資料)
# ==========================================
print(f"🧹 清洗前總資料筆數: {len(df_history)} 筆")

# 新增欄位
df_history['scrape_timestamp'] = df_history['source_file'].str.extract(r'_(\d{8}_\d{6})\.json')
df_history['scrape_time'] = pd.to_datetime(df_history['scrape_timestamp'], format='%Y%m%d_%H%M%S', errors='coerce')
df_history['update_date'] = df_history['update_time'].dt.date
df_history['hour'] = df_history['update_time'].dt.hour
df_history['weekday'] = df_history['update_time'].dt.dayofweek

# 計算每一筆資料的「資料老化時間 (天數)」
df_history['data_age_days'] = (df_history['scrape_time'] - df_history['update_time']).dt.total_seconds() / 86400

# 1. 判定每筆「紀錄」的有效性 (Row-level 條件判斷)
# 條件 A：時間凍結 (超過 1 天未更新)
is_frozen = df_history['data_age_days'] > 1
# 條件 B：無效營運 (可借與可還同時為 0)
is_dead = (df_history['sbi_quantity'] == 0) & (df_history['bemp'] == 0)

# 將異常紀錄標註起來
df_history['is_bad_record'] = (is_frozen | is_dead).astype(int)

# 計算受到異常紀錄影響的「不重複站點數」與「總異常筆數」
bad_stations_count = df_history[df_history['is_bad_record'] == 1]['sno'].nunique()
bad_records_count = df_history['is_bad_record'].sum()

print(f"🚫 發現 {bad_stations_count} 個站點 共 {bad_records_count} 筆異常紀錄 (時間凍結或暫停營運)，準備剔除！")

# 3. 執行過濾 (只保留正常紀錄)
df_filtered = df_history[df_history['is_bad_record'] == 0].copy()

# 4. 🔄 去除重複資料 (Deduplication)
# 依據站點代號(sno)與API更新時間(update_time)進行去重，保留檔案撈取時間(scrape_time)最新的一筆
df_clean = df_filtered.sort_values('scrape_time').drop_duplicates(
    subset=['sno', 'update_time'],
    keep='last'
).copy()

duplicate_records_count = len(df_filtered) - len(df_clean)
print(f"🔄 發現 {duplicate_records_count} 筆重複紀錄 (站點代號(sno)與API更新時間(update_time)進行去重)，準備剔除！")

# 重新計算負載率與狀態燈號
df_clean['load_factor'] = (df_clean['sbi_quantity'] / df_clean['tot_quantity']) * 100
df_clean['status_color'] = 'green'
df_clean.loc[(df_clean['sbi_quantity'] == 0) | (df_clean['load_factor'] < 10), 'status_color'] = 'red'
df_clean.loc[(df_clean['bemp'] == 0) | (df_clean['load_factor'] > 90), 'status_color'] = 'orange'

# 把輔助計算的欄位拿掉
df_clean = df_clean.drop(columns=['data_age_days', 'scrape_timestamp', 'is_bad_record'], errors='ignore')

print(f"✨ 最終清洗與去重後總資料筆數: {len(df_clean)} 筆 (資料集已完全淨化！)")
print("="*45)
display(df_clean.info())
print("="*45)
display(df_clean.head())

🧹 清洗前總資料筆數: 1008976 筆
🚫 發現 87 個站點 共 19466 筆異常紀錄 (時間凍結或暫停營運)，準備剔除！
🔄 發現 566186 筆重複紀錄 (站點代號(sno)與API更新時間(update_time)進行去重)，準備剔除！
✨ 最終清洗與去重後總資料筆數: 423324 筆 (資料集已完全淨化！)
<class 'pandas.core.frame.DataFrame'>
Index: 423324 entries, 23206 to 1008975
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   city          423324 non-null  object        
 1   sno           423324 non-null  object        
 2   sna           423324 non-null  object        
 3   sarea         423324 non-null  object        
 4   tot_quantity  423324 non-null  int64         
 5   sbi_quantity  423324 non-null  int64         
 6   bemp          423324 non-null  int64         
 7   lat           423324 non-null  float64       
 8   lng           423324 non-null  float64       
 9   update_time   423324 non-null  datetime64[ns]
 10  source_file   423324 non-null  object        
 11  scrape_time   423324 non-null  datetime64[ns]
 12  updat

None

,city,sno,sna,sarea,tot_quantity,sbi_quantity,bemp,lat,lng,update_time,source_file,scrape_time,update_date,hour,weekday,load_factor,status_color
23206,New Taipei,500229120,YouBike2.0_捷運幸福站(2號出口),新莊區,35,15,20,25.04955,121.46028,2026-06-16 12:31:02,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,12,1,42.857143,green
23205,New Taipei,500229039,YouBike2.0_輔仁大學附設醫院,新莊區,41,11,14,25.03927,121.43100,2026-06-16 12:31:02,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,12,1,26.829268,green
23195,New Taipei,500220024,YouBike2.0_台麗街13巷,泰山區,22,3,19,25.03829,121.42464,2026-06-16 12:31:02,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,12,1,13.636364,green
23194,New Taipei,500218212,YouBike2.0_浮洲合宜住宅(合宜一合宜路口),板橋區,22,2,20,24.99536,121.44602,2026-06-16 12:31:02,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,12,1,9.090909,red
23193,New Taipei,500218205,YouBike2.0_松柏街50巷90弄口,板橋區,12,4,8,25.02833,121.47621,2026-06-16 12:31:02,NTPC_20260616_124141.json,2026-06-16 12:41:41,2026-06-16,12,1,33.333333,green


## 輸出黑名單站點的完整歷史資料表格

In [3]:
# ==========================================
# 📊 輸出「去重複」後黑名單站點的清單表格
# ==========================================

if len(blacklist) > 0:
    # 1. 從歷史資料庫中，篩選出所有屬於黑名單的資料
    df_blacklist_details = df_history[df_history['sno'].isin(blacklist)].copy()

    # 2. 🔄 關鍵修正：依照時間排序，並針對「站點代號」去重複，只保留最後一次撈取的異常狀態
    df_blacklist_unique = df_blacklist_details.sort_values(by='scrape_time').drop_duplicates(
        subset=['sno'],
        keep='last'
    )

    # 3. 依照「縣市」與「站點代號」排序，讓清單井然有序
    df_blacklist_unique = df_blacklist_unique.sort_values(by=['city', 'sno']).reset_index(drop=True)

    # 4. 挑選核心展示欄位並標準化中文名稱
    display_columns = [
        'city', 'sno', 'sna', 'sarea', 'tot_quantity',
        'sbi_quantity', 'bemp', 'update_time', 'scrape_time'
    ]

    df_blacklist_table = df_blacklist_unique[display_columns].rename(columns={
        'city': '縣市',
        'sno': '站點代號',
        'sna': '站點名稱',
        'sarea': '行政區',
        'tot_quantity': '總車位',
        'sbi_quantity': '可借車數',
        'bemp': '可還空位',
        'update_time': 'API最後更新時間',
        'scrape_time': 'Data Lake最後撈取時間'
    })

    # 5. 列印表格
    print(f"🚨 黑名單異常站點清單 (已去重複，共計精準 {len(df_blacklist_table)} 個站點)：")

    # 設定顯示列數上限，確保能完全展開不被隱藏
    with pd.option_context('display.max_rows', 100, 'display.max_columns', None, 'display.width', 1000):
        display(df_blacklist_table)

else:
    print("✨ 太棒了！目前歷史數據中沒有發現任何符合黑名單條件的異常站點。")

NameError: name 'blacklist' is not defined

#全域總覽儀表板 (Macro Overview Dashboard)

In [4]:
# ==========================================
# 📊 Data Lake 24小時全覽儀表板：雙北分行 KPI ＋ 異常集中堆疊圖
# ==========================================

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. 🔍 資料前處理與特徵工程
# 從檔名中萃取精準的實際撈取時間 (HH:MM)
df_history['scrape_timestamp'] = df_history['source_file'].str.extract(r'_(\d{8}_\d{6})\.json')
df_history['scrape_time'] = pd.to_datetime(df_history['scrape_timestamp'], format='%Y%m%d_%H%M%S', errors='coerce')

# 將時間格式化為「小時:分鐘」，作為 24 小時軸的乾淨標籤
df_history = df_history.sort_values('scrape_time')
df_history['time_axis'] = df_history['scrape_time'].dt.strftime('%H:%M')

# 精準計算實際快照次數
total_snapshots = df_history['scrape_timestamp'].nunique()

# 擷取最後一次紀錄（最新快照）進行雙北分流統計
latest_snapshot_id = df_history['scrape_timestamp'].max()

# 2. 🗂️ 建立即時動態分類標籤 (包含黑名單)
# 將原始歷史資料標註黑名單狀態
df_history['final_status'] = df_history['status_color']
df_history.loc[df_history['sno'].isin(blacklist), 'final_status'] = 'blacklist'

# 擷取最新快照進行雙北分流 KPI 計算
df_latest_all = df_history[df_history['scrape_timestamp'] == latest_snapshot_id]

# --- 台北市即時數據統計 ---
tpe_latest_all = df_latest_all[df_latest_all['city'] == 'Taipei']
tpe_total = len(tpe_latest_all)
tpe_red = len(tpe_latest_all[tpe_latest_all['final_status'] == 'red'])
tpe_blacklist = len(tpe_latest_all[tpe_latest_all['final_status'] == 'blacklist'])

# --- 新北市即時數據統計 ---
ntpc_latest_all = df_latest_all[df_latest_all['city'] == 'New Taipei']
ntpc_total = len(ntpc_latest_all)
ntpc_red = len(ntpc_latest_all[ntpc_latest_all['final_status'] == 'red'])
ntpc_blacklist = len(ntpc_latest_all[ntpc_latest_all['final_status'] == 'blacklist'])

# 3. 📈 準備 24 小時時序異常趨勢資料 (排除 green)
trend_df = df_history.groupby(['time_axis', 'final_status']).size().unstack(fill_value=0).reset_index()
for col in ['orange', 'red', 'blacklist']:
    if col not in trend_df.columns:
        trend_df[col] = 0

# 4. 🎨 建立全新的 3x3 結構化佈局 (Row 1: 台北, Row 2: 新北, Row 3: 異常長條圖)
fig_operator = make_subplots(
    rows=3, cols=3,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}], # Row 1: 台北一行
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}], # Row 2: 新北一行
        [{"type": "xy", "colspan": 3}, None, None]                             # Row 3: 24h趨勢圖
    ],
    row_heights=[0.15, 0.15, 0.70],
    vertical_spacing=0.08
)

# --- 🟦 第一行：台北市專屬即時營運看板 ---
fig_operator.add_trace(go.Indicator(
    mode="number", value=tpe_total,
    title={"text": "🏙️ 台北市：總營運站數", "font": {"size": 14, "color": "#1F77B4"}},
    number={'valueformat': ",", 'font': {'size': 32}}
), row=1, col=1)

fig_operator.add_trace(go.Indicator(
    mode="number", value=tpe_red,
    title={"text": "🔴 台北市：即時缺車紅區", "font": {"size": 14, "color": "#FF4136"}},
    number={'font': {'color': '#FF4136', 'size': 32}}
), row=1, col=2)

fig_operator.add_trace(go.Indicator(
    mode="number", value=tpe_blacklist,
    title={"text": "⚫ 台北市：故障黑名單", "font": {"size": 14, "color": "#7F7F7F"}},
    number={'font': {'color': '#7F7F7F', 'size': 32}}
), row=1, col=3)

# --- 🟩 第二行：新北市專屬即時營運看板 ---
fig_operator.add_trace(go.Indicator(
    mode="number", value=ntpc_total,
    title={"text": "🌲 新北市：總營運站數", "font": {"size": 14, "color": "#2CA02C"}},
    number={'valueformat': ",", 'font': {'size': 32}}
), row=2, col=1)

fig_operator.add_trace(go.Indicator(
    mode="number", value=ntpc_red,
    title={"text": "🔴 新北市：即時缺車紅區", "font": {"size": 14, "color": "#FF4136"}},
    number={'font': {'color': '#FF4136', 'size': 32}}
), row=2, col=2)

fig_operator.add_trace(go.Indicator(
    mode="number", value=ntpc_blacklist,
    title={"text": "⚫ 新北市：故障黑名單", "font": {"size": 14, "color": "#7F7F7F"}},
    number={'font': {'color': '#7F7F7F', 'size': 32}}
), row=2, col=3)

# --- 📊 第三行：全網異常聚焦疊加圖 (灰色底座 ＋ 排除綠燈) ---
# 依照：黑名單(灰底) -> 滿站(橘) -> 嚴重缺車(紅) 的順序往上疊加，層次鮮明
anomaly_configs = [
    {'status': 'blacklist', 'color': '#A0A0A0', 'name': '⚫ 營運黑名單 (故障/幽靈站)'},
    {'status': 'orange', 'color': '#FF851B', 'name': '🟠 滿站警戒區 (無位可還)'},
    {'status': 'red', 'color': '#FF4136', 'name': '🔴 缺車警戒區 (無車可借)'}
]

for cfg in anomaly_configs:
    fig_operator.add_trace(go.Bar(
        x=trend_df['time_axis'],
        y=trend_df[cfg['status']],
        name=cfg['name'],
        marker_color=cfg['color'],
        hovertemplate="時間: %{x}<br>%{name}: %{y} 站<extra></extra>"
    ), row=3, col=1)

# 5. ⚙️ 全局美化優化
fig_operator.update_layout(
    title_text=f"🚀 大臺北 YouBike 2.0 決策戰情室 (歷史快照計: {df_history['scrape_timestamp'].nunique()} 次)",
    title_font_size=22,
    barmode='stack',
    hovermode="x unified",
    height=800,
    margin=dict(t=120, b=40, l=60, r=60),
    legend=dict(orientation="h", yanchor="bottom", y=0.68, xanchor="right", x=1), # 巧妙安放圖例位置
    xaxis3=dict(title="當日收集進度時間軸 (已過濾綠燈正常資料)", type='category'),
    yaxis3=dict(title="異常站點數量")
)

fig_operator.show()

KeyError: 'status_color'

## 探索性資料分析 (EDA - Exploratory Data Analysis

In [ ]:
# ==========================================
# 📊 基本現況總覽：時間跨度、數值分佈與最新快照健康度
# ==========================================
import pandas as pd

print("=== 🌟 1. 歷史資料基本資訊 (Time Span & Scale) ===")
# 計算時間跨度與規模
min_time = df_clean['update_time'].min()
max_time = df_clean['update_time'].max()
unique_stations = df_clean['sno'].nunique()
total_records = len(df_clean)

summary_info = pd.DataFrame({
    '指標': ['最早更新時間', '最新更新時間', '總觀測營運站點數', '總乾淨數據筆數'],
    '數值': [
        min_time.strftime('%Y-%m-%d %H:%M:%S'),
        max_time.strftime('%Y-%m-%d %H:%M:%S'),
        f"{unique_stations:,} 站 (已扣除黑名單)",
        f"{total_records:,} 筆"
    ]
})
display(summary_info)

print("\n=== 📈 2. 全域營運數值分佈 (Global Metrics) ===")
# 挑選數值型欄位進行統計描述 (describe)
desc_cols = ['tot_quantity', 'sbi_quantity', 'bemp', 'load_factor']
df_desc = df_clean[desc_cols].describe().round(2).T
df_desc = df_desc.rename(index={
    'tot_quantity': '總車位數',
    'sbi_quantity': '可借車數',
    'bemp': '可還空位',
    'load_factor': '負載率(%)'
})
# 顯示平均數、標準差、最大最小值與四分位數
display(df_desc[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

print("\n=== 🏥 3. 最新快照健康度 (Latest Snapshot Health) ===")
# 取得 Data Lake 中最後一次撈取的時間截點
latest_time = df_clean['scrape_time'].max()
df_latest_clean = df_clean[df_clean['scrape_time'] == latest_time]

# 計算各燈號數量 (使用 reindex 確保三種燈號都會顯示，即使數量為 0)
health_status = df_latest_clean['status_color'].value_counts().reindex(['green', 'orange', 'red'], fill_value=0).reset_index()
health_status.columns = ['燈號狀態', '站點數量']

# 加入中文註解與比例
status_map = {'green': '🟢 正常區間', 'orange': '🟠 滿站警戒 (無位可還)', 'red': '🔴 缺車警戒 (無車可借)'}
health_status['狀態說明'] = health_status['燈號狀態'].map(status_map)
health_status['佔比(%)'] = (health_status['站點數量'] / len(df_latest_clean) * 100).round(1)

display(health_status[['狀態說明', '站點數量', '佔比(%)']])

## 雙北 YouBike 2.0 站點容量資源分佈地圖 (Station Capacity Map)

從三個空間維度來看出故事：

- 交通樞紐的「巨型大動脈」：
那些面積大到誇張的藍色圓圈，通常會極度精準地黏在捷運轉乘站、台鐵/高鐵站、以及大學城（如台大公館生活圈）的周邊。這代表微笑單車在都市計畫中，是扮演標準的「大眾運輸最後一哩路（Last-Mile Connector）」。

- 社區內部的「微血管分佈」：
相較之下，住宅區、巷弄、小公園周邊則佈滿了非常密集、但藍圈極小的「微型站（10~15 柱）」。這能展現 YouBike 2.0 無須牽電線的靈活特性，成功把服務觸角塞進城市的死角。

- 調度與營運瓶頸預警（Bottleneck Alert）：
藍色圓圈越大，代表它在通勤尖峰時間能吐出、或吞下的車流量越驚人！ 這張圖直接點名了哪些站點是未來的「調度一線戰場」。這些大圈圈只要一壞掉或被塞滿，全網的流動性就會瞬間卡死。

In [ ]:
# ==========================================
# 🗺️ 雙北 YouBike 2.0 站點容量資源分佈地圖 (Station Capacity Map)
# ==========================================

import plotly.graph_objects as go
import pandas as pd

# 1. 萃取最新一筆快照的乾淨資料 (避免黑名單干擾)
latest_snapshot_time = df_clean['scrape_time'].max()
df_map_data = df_clean[df_clean['scrape_time'] == latest_snapshot_time].copy()

# 2. 初始化 Plotly 地圖物件
fig_capacity = go.Figure()

# 3. 圖層 A：藍色半透明圓圈 (圓圈半徑正比於總車位數)
# 透過 marker_size 控制車位映射，並設定 hover 顯示站點明細
fig_capacity.add_trace(go.Scattermapbox(
    lat=df_map_data['lat'],
    lon=df_map_data['lng'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=df_map_data['tot_quantity'], # 核心設計：大小正比於場站總停車格
        sizemode='diameter',
        sizeref=2.5,                     # 縮放因子，調整藍圈在網頁上的視覺大小 (數字越大圈越小)
        color='#1F77B4',                 # 經典商務藍
        opacity=0.4                      # 半透明疊加，即使大站堆疊也能看清底層
    ),
    name='場站總容量 (車位規模)',
    text=df_map_data['sna'],
    hovertemplate="<b>%{text}</b><br>" +
                  "縣市: " + df_map_data['city'] + "<br>" +
                  "行政區: " + df_map_data['sarea'] + "<br>" +
                  "總車位數: %{marker.size} 柱<extra></extra>"
))

# 4. 圖層 B：中心精準錨點 (微小的綠色實心點，標註確切位置)
fig_capacity.add_trace(go.Scattermapbox(
    lat=df_map_data['lat'],
    lon=df_map_data['lng'],
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=4,                          # 微小的精準定位點
        color='#2ECC40',                 # 活力綠
        opacity=1.0
    ),
    name='站點精準幾何位置',
    hoverinfo='skip'                     # 略過 hover 免得干擾藍圈的 Tooltip
))

# 5. 地圖版面與底圖配置 (採用極簡白底圖，最突顯數據)
fig_capacity.update_layout(
    title={
        'text': f"🗺️ 大臺北 YouBike 2.0 基礎基礎設施容量分佈圖<br><span style='font-size:12px;color:grey;'>資料快照截點: {latest_snapshot_time.strftime('%Y-%m-%d %H:%M')} | 觀測站點數: {len(df_map_data)} 站</span>",
        'y':0.95, 'x':0.05, 'xanchor': 'left', 'yanchor': 'top'
    },
    mapbox_style="carto-positron",       # 極簡白色底圖，商務感極強
    mapbox_zoom=11.5,                    # 縮放等級，剛好包覆雙北核心生活圈
    mapbox_center={"lat": 25.04, "lon": 121.51}, # 聚焦雙北盆地中心
    height=750,
    margin={"r":0, "t":80, "l":0, "b":0},
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top", y=0.02,
        xanchor="right", x=0.98,
        bgcolor="rgba(255, 255, 255, 0.8)"
    )
)

fig_capacity.show()

### Folium 版本
Folium 的多圖層機制：

1.藍色圓圈圖層：圓圈半徑正比於場站總車位數（tot_quantity），並設定了不透明度，即使大站堆疊也能看清底層。

2.中心錨點圖層：微小的綠色實心點標註確切位置。

這樣的視覺化可以讓我們看出三大策略觀察點：

1.大眾運輸最後一哩路：最大的圓圈（樞紐）極度精準地黏在捷運轉乘站、台鐵/高鐵站周邊。

2.社區內部的「微血管」：住宅區、巷弄周邊佈滿密集但圓圈極小的微型站。

3.調度瓶頸預警：藍圈越大，代表通勤尖峰時段能吐出或吞下的車流量越驚人，這些圓圈只要壞掉或被塞滿，全網的流動性就會瞬间卡死。

In [ ]:
# ==========================================
# 🗺️ Folium 版：雙北 YouBike 2.0 站點容量資源分佈地圖
# ==========================================
import folium

# 1. 確保取用最新一筆快照的乾淨資料
latest_snapshot_time = df_clean['scrape_time'].max()
df_map_data = df_clean[df_clean['scrape_time'] == latest_snapshot_time].copy()

print(f"⏳ 正在繪製 Folium 容量地圖，共 {len(df_map_data)} 個站點，這可能需要幾秒鐘...")

# 2. 初始化 Folium 地圖 (使用 cartodbpositron 極簡白底圖，凸顯資料)
m_folium_capacity = folium.Map(
    location=[25.04, 121.51],
    zoom_start=12,
    tiles='cartodbpositron'
)

# 3. 迭代資料，將每一站畫上地圖
for idx, row in df_map_data.iterrows():

    # 設定滑鼠懸停顯示的 Tooltip 資訊
    tooltip_text = (
        f"<b>{row['sna']}</b><br>"
        f"縣市: {row['city']}<br>"
        f"行政區: {row['sarea']}<br>"
        f"總車位數: {row['tot_quantity']} 柱"
    )

    # 圖層 A：藍色半透明圓圈 (代表總車位容量)
    # Folium 的 radius 在 CircleMarker 單位是 pixel，我們除以 1.5 作為視覺縮放因子
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=row['tot_quantity'] / 1.5,
        color='#1F77B4',
        weight=1,
        fill=True,
        fill_color='#1F77B4',
        fill_opacity=0.4,
        tooltip=tooltip_text
    ).add_to(m_folium_capacity)

    # 圖層 B：綠色精準中心點 (定位確切的幾何點位)
    folium.CircleMarker(
        location=[row['lat'], row['lng']],
        radius=1.5,
        color='#2ECC40',
        weight=1,
        fill=True,
        fill_color='#2ECC40',
        fill_opacity=1.0
    ).add_to(m_folium_capacity)

print("✅ Folium 地圖繪製完成！")

# 4. 渲染地圖
m_folium_capacity